# Music as a Sentiment Barometer
## Notebook 4 — Results, Write-Up & Dashboard

This notebook consolidates findings from Notebooks 1–3 into a structured write-up, answers the core hypothesis, acknowledges limitations, and produces an interactive HTML dashboard exportable to GitHub.

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('Imports successful!')

In [ ]:
# ── LOAD DATA ─────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

PATH = '/content/drive/MyDrive/side_proj/'

full = pd.read_csv(PATH + 'final_dataset_full.csv')
full['date'] = pd.to_datetime(full['date'])
full_clean = full.dropna(subset=['misery_index']).copy()

sent = pd.read_csv(PATH + 'final_dataset_sentiment.csv')
sent['date'] = pd.to_datetime(sent['date'])
sent_clean = sent.dropna(subset=['misery_index', 'michigan_sentiment']).copy()

# Load raw merged Billboard + audio features for song table
billboard = pd.read_csv(PATH + 'billboard_audio_merged.csv')
billboard['chart_date'] = pd.to_datetime(billboard['chart_date'])

print(f'Datasets loaded successfully')
print(f'Economic data  : {len(full_clean):,} months ({full_clean.date.min()\
                                                      .year}–{full_clean.date.max().year})')
print(f'Billboard data : {len(billboard):,} chart entries')

---
## 1. Project Recap

This project extends Larson (2022)'s *The Lure of Musical Comfort* by introducing consumer sentiment as a competing predictor of musical taste alongside hard economic indicators. Using 66 years of Billboard Hot 100 chart data (1958–2024), Spotify audio features from the TidyTuesday archive, and five economic series from FRED, we tested whether the sonic character of popular music responds more strongly to *what the economy is* (Okun Misery Index) or *how people feel about it* (Michigan Consumer Sentiment Index).

Four models were estimated: a Larson replication (Model A), a sentiment-only model (Model B), a combined horse race (Model C), and a lag analysis (Model D). Ridge regression was run as a robustness check to account for multicollinearity between the two predictors.

---
## 2. Key Findings

**The Misery Index drives the emotional character of music.**  
Loudness (R²=0.253), valence (R²=0.249), and speechiness (R²=0.248) are the three strongest associations in the dataset, all dominated by the Misery Index and confirmed as robust by both OLS and Ridge. During periods of objective economic hardship, Americans gravitate toward happier, quieter, and less vocal-heavy music — consistent with Larson (2022)'s comfort-seeking thesis.

**Michigan Sentiment drives the rhythmic and energetic character of music.**  
Tempo (R²=0.109), instrumentalness (R²=0.107), energy (R²=0.034), and acousticness (R²=0.022) are more strongly associated with perceived economic conditions than with hard data. When consumer confidence falls, music slows down, becomes more acoustic, and loses energy — a more subdued, inward response than the emotional comfort-seeking pattern above.

**Neither indicator alone tells the full story.**  
The combined model (Model C) consistently outperforms both individual models across all features, confirming that the Misery Index and Michigan Sentiment capture independent, additive information. The two indicators are driving different sonic dimensions simultaneously.

**Tempo is the clearest evidence of a delayed, sentiment-driven response.**  
The lag analysis found that tempo's peak correlation with Michigan Sentiment occurs at lag 5 — meaning consumer confidence shifts show up in the pace of charting music approximately 5 months later. This is consistent with the music production timeline and provides the strongest directional evidence in the analysis.

**Danceability and energy do not support the simple recession pop narrative.**  
Despite the cultural narrative around recession pop being built on danceability and high-energy music, both features show near-zero associations across all models at the monthly average level. The relationship, if it exists, is likely non-linear, genre-specific, or driven by factors outside the scope of this analysis.

---
## 3. Answering the Core Hypothesis

The original hypothesis was that Michigan Consumer Sentiment would outperform the Misery Index as a predictor of audio features. The results give a more nuanced answer: **it depends on which sonic dimension you are measuring.**

The Misery Index is the stronger predictor of the *emotional* character of music — valence, loudness, and speechiness — while Michigan Sentiment is the stronger predictor of the *rhythmic and energetic* character — tempo, acousticness, and energy. Neither fully dominates the other. The most accurate conclusion is that actual economic pain and perceived economic anxiety are two distinct psychological states that manifest in different aspects of musical preference simultaneously.

---
## 4. Limitations

**Match rate:** The merge between Billboard chart data and TidyTuesday audio features achieved an 81.1% match rate, meaning approximately 19% of chart entries had no audio feature data and were excluded from the analysis. Songs from earlier decades (pre-1970) are likely underrepresented in the Spotify catalog.

**Audio feature coverage:** The TidyTuesday dataset covers songs through 2021. Post-2021 chart entries from the HipsterVizNinja source may have reduced audio feature coverage, which could affect the reliability of findings for the most recent period.

**Low R² values:** The highest R² in the dataset is 0.253. Economic conditions explain some but not most of the variation in musical taste. Technology shifts, streaming algorithms, demographic changes, and individual artist trends all contribute to music popularity in ways this model does not capture.

**Correlation, not causation:** All findings are associational. The analysis cannot establish that economic conditions *cause* shifts in musical taste — only that they move together in a statistically meaningful way.

**Monthly aggregation:** Averaging audio features across all Hot 100 songs each month smooths over significant within-month variation and genre-specific dynamics that a more granular analysis might reveal.

---
## 5. Resume Bullet

> **Music as a Sentiment Barometer: Recession Pop & Consumer Confidence Model** — Python, FRED, GitHub  
> Analysed 66 years of Billboard Hot 100 audio features (valence, tempo, loudness) against US economic indicators using OLS regression and Ridge regularisation. Found that the Okun Misery Index predicts the *emotional* character of popular music (loudness R²=0.25, valence R²=0.25) while Michigan Consumer Sentiment predicts its *rhythmic* character (tempo R²=0.11), with a 5-month lag — supporting the hypothesis that music reflects perceived economic anxiety rather than hard economic data alone.

---
## 6. Next Steps

- **Genre-level analysis:** Disaggregate the analysis by genre (pop, hip-hop, country, R&B) — Larson (2022) found that the direction of effects reverses for country music, which would be worth replicating with this extended dataset
- **Non-linear models:** Test whether polynomial regression or a time-series model (VAR, ARIMA) captures the economy-music relationship more accurately than OLS
- **Lyrical sentiment:** Add NLP-based lyrical sentiment analysis as a third musical dimension alongside audio features
- **Cross-country comparison:** Extend to UK or Germany charts to test whether the patterns are US-specific or universal
- **Real-time dashboard:** Connect the dashboard to live FRED API data to produce a continuously updated economic music barometer

---
## 7. Interactive Dashboard

The dashboard below allows exploration of the full dataset by date range. Select a period to see:
- **Economic status** — Misery Index, Michigan Sentiment, and recession bands
- **Sonic fingerprint** — radar chart of average audio features vs. all-time baseline
- **Top 5 songs** — highest charting songs from the selected period

Export as a standalone HTML file for GitHub hosting.

In [6]:
# ── DASHBOARD ─────────────────────────────────────────────────────────────────
!pip install plotly --quiet

# Pre-labeled eras
ERAS = [
    {'name': '1973-75 Oil Crisis',  'start': '1973-11-01', 'end': '1975-03-01', 'color': 'rgba(255,200,100,0.15)'},
    {'name': '1990-91 Recession',   'start': '1990-07-01', 'end': '1991-03-01', 'color': 'rgba(255,150,150,0.15)'},
    {'name': '2008 Great Recession','start': '2007-12-01', 'end': '2009-06-01', 'color': 'rgba(200,100,255,0.15)'},
    {'name': '2020 COVID',          'start': '2020-02-01', 'end': '2020-04-01', 'color': 'rgba(100,200,255,0.15)'},
]

# All-time audio feature baseline
RADAR_FEATURES = ['valence', 'danceability', 'energy', 'acousticness', 'speechiness']
BASELINE = full_clean[RADAR_FEATURES].mean().to_dict()

RADAR_MIN = {f: full_clean[f].min() for f in RADAR_FEATURES}
RADAR_MAX = {f: full_clean[f].max() for f in RADAR_FEATURES}

def normalise(val, feat):
    return (val - RADAR_MIN[feat]) / (RADAR_MAX[feat] - RADAR_MIN[feat])

baseline_norm = [normalise(BASELINE[f], f) for f in RADAR_FEATURES]

# Pre-compute data
econ_data = full_clean[['date', 'misery_index', 'michigan_sentiment', 'recession',
                          'unemployment', 'inflation'] + RADAR_FEATURES].copy()
econ_data['date_str'] = econ_data['date'].dt.strftime('%Y-%m')

songs_data = billboard[[
    'chart_date', 'song', 'performer', 'peak_position',
    'time_on_chart', 'valence', 'danceability'
]].copy()
songs_data['date_str'] = songs_data['chart_date'].dt.strftime('%Y-%m')

# Reduce to yearly averages for econ data (smaller payload)
econ_data = econ_data.groupby(econ_data['date'].dt.year).first().reset_index(drop=True)
econ_data['date_str'] = econ_data['date'].dt.strftime('%Y-%m')

# Limit songs to top 500 by time_on_chart (covers the most notable songs)
songs_data = songs_data.sort_values('time_on_chart', ascending=False).head(500)

import json

econ_json   = econ_data.to_json(orient='records', date_format='iso')
songs_json  = songs_data.dropna(subset=['peak_position']).to_json(orient='records', date_format='iso')
eras_json   = json.dumps(ERAS)
radar_feats = json.dumps(RADAR_FEATURES)
base_norm   = json.dumps(baseline_norm)
radar_min   = json.dumps(RADAR_MIN)
radar_max   = json.dumps(RADAR_MAX)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Music as a Sentiment Barometer</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #0f1117; color: #e0e0e0; padding: 20px; }}
  h1 {{ font-size: 1.6em; color: #1DB954; margin-bottom: 4px; }}
  h2 {{ font-size: 1.0em; color: #aaa; font-weight: normal; margin-bottom: 20px; }}
  .controls {{ background: #1a1d27; border-radius: 10px; padding: 20px; margin-bottom: 20px; }}
  .controls label {{ font-size: 0.85em; color: #aaa; display: block; margin-bottom: 6px; }}
  .slider-row {{ display: flex; align-items: center; gap: 12px; flex-wrap: wrap; }}
  input[type=range] {{ flex: 1; min-width: 200px; accent-color: #1DB954; }}
  .date-label {{ font-size: 0.9em; color: #1DB954; font-weight: bold; min-width: 70px; }}
  .grid-3 {{ display: grid; grid-template-columns: 2fr 1fr; gap: 16px; }}
  .card {{ background: #1a1d27; border-radius: 10px; padding: 16px; }}
  .card h3 {{ font-size: 0.85em; color: #aaa; margin-bottom: 10px; text-transform: uppercase; letter-spacing: 0.05em; }}
  .stat-row {{ display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 16px; }}
  .stat {{ background: #252836; border-radius: 8px; padding: 10px 14px; flex: 1; min-width: 100px; }}
  .stat .val {{ font-size: 1.4em; font-weight: bold; color: #1DB954; }}
  .stat .lbl {{ font-size: 0.75em; color: #888; margin-top: 2px; }}
  .recession-badge {{ display: inline-block; background: #E63946; color: white; border-radius: 4px; padding: 2px 8px; font-size: 0.75em; }}
  .expansion-badge {{ display: inline-block; background: #1DB954; color: white; border-radius: 4px; padding: 2px 8px; font-size: 0.75em; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 0.82em; }}
  th {{ text-align: left; color: #888; font-weight: normal; padding: 6px 8px; border-bottom: 1px solid #2a2d3a; }}
  td {{ padding: 6px 8px; border-bottom: 1px solid #1e2130; }}
  tr:hover td {{ background: #252836; }}
  .rank {{ color: #1DB954; font-weight: bold; }}
  .chart-container {{ width: 100%; }}
  .explain-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 16px; margin-bottom: 20px; }}
  .explain-card {{ background: #1a1d27; border-radius: 10px; padding: 16px; }}
  .explain-title {{ font-size: 0.75em; text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 6px; font-weight: bold; }}
  .explain-body {{ font-size: 0.85em; color: #ccc; line-height: 1.6; }}
  @media (max-width: 700px) {{
    .grid-3, .explain-grid {{ grid-template-columns: 1fr; }}
  }}
</style>
</head>
<body>

<h1> Music as a Sentiment Barometer</h1>
<h2>66 years of Billboard Hot 100 audio features mapped against US economic conditions (1958-2024)</h2>

<div class="explain-grid">
  <div class="explain-card" style="border-left: 3px solid #E63946;">
    <div class="explain-title" style="color:#E63946;">Okun Misery Index</div>
    <div class="explain-body">
      A hard economic indicator calculated as <strong>unemployment rate + inflation rate</strong>.
      It measures the objective state of the economy — what is actually happening regardless of
      how people feel about it. Research shows it is the strongest predictor of the
      <em>emotional</em> character of popular music: higher misery is associated with
      happier, quieter, and less vocal-heavy songs as people seek comfort through music.
    </div>
  </div>
  <div class="explain-card" style="border-left: 3px solid #457B9D;">
    <div class="explain-title" style="color:#457B9D;"> Michigan Consumer Sentiment</div>
    <div class="explain-body">
      A survey-based measure of how confident Americans feel about their personal finances
      and the broader economy. Unlike the Misery Index, it captures <em>perceived</em>
      economic anxiety — how bad things feel, not just how bad they are. It is the stronger
      predictor of the <em>rhythmic</em> character of music: lower sentiment is associated
      with slower tempo and lower energy songs, with a ~5 month lag reflecting the
      music production timeline.
    </div>
  </div>
</div>

<div class="controls">
  <label>Select Date Range</label>
  <div class="slider-row">
    <span class="date-label" id="startLabel">1958</span>
    <input type="range" id="startSlider" min="0" max="0" value="0" oninput="updateDashboard()">
    <span style="color:#555">&#8594;</span>
    <input type="range" id="endSlider" min="0" max="0" value="0" oninput="updateDashboard()">
    <span class="date-label" id="endLabel">2024</span>
  </div>
</div>

<div class="stat-row" id="statsRow"></div>

<div class="grid-3">
  <div class="card">
    <h3>Economic Timeline</h3>
    <div class="chart-container" id="econChart"></div>
  </div>
  <div class="card">
    <h3>Sonic Fingerprint vs. All-Time Average</h3>
    <div class="chart-container" id="radarChart"></div>
  </div>
</div>

<div class="card" style="margin-top:16px">
  <h3>Top 5 Songs of the Period</h3>
  <div id="songsTable"></div>
</div>

<script>
const econData   = {econ_json};
const songsData  = {songs_json};
const eras       = {eras_json};
const radarFeats = {radar_feats};
const baselineNorm = {base_norm};
const radarMin   = {radar_min};
const radarMax   = {radar_max};

const dates = econData.map(d => d.date_str);
const startSlider = document.getElementById('startSlider');
const endSlider   = document.getElementById('endSlider');
startSlider.max = dates.length - 1;
endSlider.max   = dates.length - 1;
endSlider.value = dates.length - 1;

function normalise(val, feat) {{
  return (val - radarMin[feat]) / (radarMax[feat] - radarMin[feat]);
}}

function updateDashboard() {{
  let s = parseInt(startSlider.value);
  let e = parseInt(endSlider.value);
  if (s > e) {{ endSlider.value = s; e = s; }}

  document.getElementById('startLabel').textContent = dates[s].slice(0,7);
  document.getElementById('endLabel').textContent   = dates[e].slice(0,7);

  const slice     = econData.slice(s, e + 1);
  const startDate = dates[s];
  const endDate   = dates[e];

  // ── STATS ─────────────────────────────────────────────────────────────────
  const avgMisery  = (slice.reduce((a,d) => a + (d.misery_index||0), 0) / slice.length).toFixed(1);
  const avgSent    = (slice.reduce((a,d) => a + (d.michigan_sentiment||0), 0) / slice.length).toFixed(1);
  const avgUnemp   = (slice.reduce((a,d) => a + (d.unemployment||0), 0) / slice.length).toFixed(1);
  const avgInfl    = (slice.reduce((a,d) => a + (d.inflation||0), 0) / slice.length).toFixed(1);
  const inRec      = slice.some(d => d.recession === 1);
  const avgValence = (slice.reduce((a,d) => a + (d.valence||0), 0) / slice.length).toFixed(3);
  const avgGDP = (slice.reduce((a,d) => a + (d.gdp_growth||0), 0) / slice.length).toFixed(1);

  document.getElementById('statsRow').innerHTML = `
    <div class="stat"><div class="val">${{avgMisery}}</div><div class="lbl">Avg Misery Index</div></div>
    <div class="stat"><div class="val">${{avgSent}}</div><div class="lbl">Avg Michigan Sentiment</div></div>
    <div class="stat"><div class="val">${{avgUnemp}}%</div><div class="lbl">Avg Unemployment</div></div>
    <div class="stat"><div class="val">${{avgInfl}}%</div><div class="lbl">Avg Inflation</div></div>
    <div class="stat"><div class="val">${{avgGDP}}%</div><div class="lbl">Avg GDP Growth</div></div>
    <div class="stat"><div class="val">${{avgValence}}</div><div class="lbl">Avg Valence</div></div>
    <div class="stat"><div class="val">${{inRec
      ? '<span class="recession-badge">Recession</span>'
      : '<span class="expansion-badge">Expansion</span>'}}</div><div class="lbl">Period Status</div></div>
  `;

  // ── ECON CHART ────────────────────────────────────────────────────────────
  const shapes = [];
  let inReces = false; let recStart = null;
  slice.forEach(d => {{
    if (d.recession === 1 && !inReces) {{ recStart = d.date_str; inReces = true; }}
    else if (d.recession === 0 && inReces) {{
      shapes.push({{ type:'rect', xref:'x', yref:'paper',
        x0:recStart, x1:d.date_str, y0:0, y1:1,
        fillcolor:'rgba(150,150,150,0.15)', line:{{width:0}} }});
      inReces = false;
    }}
  }});

  const annotations = [];
  eras.forEach(era => {{
    if (era.start >= startDate && era.start <= endDate) {{
      shapes.push({{ type:'rect', xref:'x', yref:'paper',
        x0:era.start, x1:era.end < endDate ? era.end : endDate,
        y0:0, y1:1, fillcolor:era.color, line:{{width:0}} }});
      annotations.push({{ x:era.start, yref:'paper', y:1.02,
        text:era.name, showarrow:false,
        font:{{size:9, color:'#aaa'}}, xanchor:'left' }});
    }}
  }});

  Plotly.react('econChart', [
    {{ x:slice.map(d=>d.date_str), y:slice.map(d=>d.misery_index),
       name:'Misery Index', line:{{color:'#E63946', width:1.5}}, mode:'lines' }},
    {{ x:slice.map(d=>d.date_str), y:slice.map(d=>d.michigan_sentiment),
       name:'Michigan Sentiment', line:{{color:'#457B9D', width:1.5}},
       mode:'lines', yaxis:'y2' }},
  ], {{
    paper_bgcolor:'#1a1d27', plot_bgcolor:'#1a1d27',
    font:{{color:'#ccc', size:10}},
    margin:{{t:30,b:40,l:50,r:50}},
    height:280,
    shapes, annotations,
    legend:{{orientation:'h', y:-0.2}},
    xaxis:{{gridcolor:'#2a2d3a', zeroline:false}},
    yaxis:{{title:'Misery Index', gridcolor:'#2a2d3a', zeroline:false}},
    yaxis2:{{title:'Sentiment', overlaying:'y', side:'right',
             gridcolor:'#2a2d3a', zeroline:false}},
  }}, {{responsive:true}});

  // ── RADAR CHART ───────────────────────────────────────────────────────────
  const periodNorm = radarFeats.map(f => {{
    const avg = slice.reduce((a,d) => a + (d[f]||0), 0) / slice.length;
    return normalise(avg, f);
  }});

  Plotly.react('radarChart', [
    {{ type:'scatterpolar', r:[...periodNorm, periodNorm[0]],
       theta:[...radarFeats, radarFeats[0]],
       fill:'toself', name:'Selected Period',
       line:{{color:'#1DB954'}}, fillcolor:'rgba(29,185,84,0.15)' }},
    {{ type:'scatterpolar', r:[...baselineNorm, baselineNorm[0]],
       theta:[...radarFeats, radarFeats[0]],
       fill:'toself', name:'All-Time Avg',
       line:{{color:'#457B9D', dash:'dot'}}, fillcolor:'rgba(69,123,157,0.1)' }},
  ], {{
    paper_bgcolor:'#1a1d27', plot_bgcolor:'#1a1d27',
    font:{{color:'#ccc', size:10}},
    polar:{{ bgcolor:'#1a1d27',
      radialaxis:{{visible:true, range:[0,1], gridcolor:'#2a2d3a', color:'#555'}},
      angularaxis:{{gridcolor:'#2a2d3a', color:'#aaa'}} }},
    margin:{{t:20,b:20,l:40,r:40}},
    height:280,
    legend:{{orientation:'h', y:-0.1}},
  }}, {{responsive:true}});

  // ── SONGS TABLE ───────────────────────────────────────────────────────────
  const periodSongs = songsData.filter(d => d.date_str >= startDate && d.date_str <= endDate);

  const songMap = {{}};
  periodSongs.forEach(d => {{
    const key = d.song + '|' + d.performer;
    if (!songMap[key] ||
        d.peak_position < songMap[key].peak_position ||
        (d.peak_position === songMap[key].peak_position &&
         d.time_on_chart > songMap[key].time_on_chart)) {{
      songMap[key] = d;
    }}
  }});

  const top5 = Object.values(songMap)
    .sort((a,b) => a.peak_position - b.peak_position || b.time_on_chart - a.time_on_chart)
    .slice(0, 5);

  if (top5.length === 0) {{
    document.getElementById('songsTable').innerHTML =
      '<p style="color:#555;font-size:0.85em">No song data for this period.</p>';
    return;
  }}

  const rows = top5.map((s,i) => `
    <tr>
      <td class="rank">#${{i+1}}</td>
      <td><strong>${{s.song}}</strong><br>
          <span style="color:#888;font-size:0.85em">${{s.performer}}</span></td>
      <td>#${{s.peak_position}}</td>
      <td>${{s.time_on_chart}} wks</td>
      <td>${{s.valence     ? s.valence.toFixed(2)     : '—'}}</td>
      <td>${{s.danceability ? s.danceability.toFixed(2) : '—'}}</td>
    </tr>`).join('');

  document.getElementById('songsTable').innerHTML = `
    <table>
      <thead><tr>
        <th>#</th><th>Song / Artist</th>
        <th>Peak</th><th>Weeks</th>
        <th>Valence</th><th>Danceability</th>
      </tr></thead>
      <tbody>${{rows}}</tbody>
    </table>`;
}}

updateDashboard();
</script>
</body>
</html>"""

# Save to Drive
OUTPUT_PATH = PATH + 'music_sentiment_dashboard.html'
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'Dashboard saved to: {OUTPUT_PATH}')
print('Upload music_sentiment_dashboard.html to GitHub — renders directly in any browser.')

Dashboard saved to: /content/drive/MyDrive/side_proj/music_sentiment_dashboard.html
Upload music_sentiment_dashboard.html to GitHub — renders directly in any browser.


---
## Preview Dashboard in Colab

Run the cell below to preview the dashboard inline in Colab before uploading to GitHub.

In [7]:
from IPython.display import HTML
with open('/content/dashboard_preview.html', 'w', encoding='utf-8') as f:
    f.write(html)
HTML(html)